# **FRAMEWORK V7 DATASET MAESTRO - DISEÑO EXPERIMENTAL**

## **M0 – Configuración**

Objetivo:

Definir los experimentos oficiales utilizados para validar el
Framework de Ingeniería de Datos y Ciencia de Datos aplicado
a la Gestión Inteligente del Recurso Hídrico.

## **M1 Objetivo**

El objetivo del Diseño Experimental es establecer el conjunto de experimentos utilizados para validar el funcionamiento del Framework, definiendo las variables objetivo, las variables predictoras, la configuración de modelado y los criterios de evaluación aplicables a cada caso de estudio.

## **M2 Variables del Dominio**

In [1]:
#==========================================================================================
# M2. VARIABLES DEL DOMINIO
#==========================================================================================

import pandas as pd

URL_VARIABLES_ML = (
    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/"
    "refs/heads/main/DATA/MASTER/C11_MASTER/"
    "Preparacion_Machine_Learning/ipm/"
    "variables_machine_learning.csv"
)

df_variables = pd.read_csv(
    URL_VARIABLES_ML,
    encoding="utf-8-sig"
)

print()
print("=" * 90)
print("VARIABLES DEL DOMINIO - DISEÑO EXPERIMENTAL")
print("=" * 90)

print()
print(f"Variables        : {len(df_variables)}")
print(f"Variables únicas : {df_variables['Variable'].nunique()}")
print(f"Duplicadas       : {df_variables['Variable'].duplicated().sum()}")
print(f"Nulos Variable   : {df_variables['Variable'].isna().sum()}")

print()
print(
    df_variables[
        ["Ranking", "Variable", "Categoria", "Importancia", "IPML", "Estado_Final"]
    ].to_string(index=False)
)


VARIABLES DEL DOMINIO - DISEÑO EXPERIMENTAL

Variables        : 10
Variables únicas : 10
Duplicadas       : 0
Nulos Variable   : 0

 Ranking                                                                 Variable              Categoria Importancia   IPML Estado_Final
       1                                                                      ONI Variabilidad Climática    Muy Alta 1.0000 Seleccionada
       2 PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS        Infraestructura    Muy Alta 1.0000 Seleccionada
       3                                                         Precipitacion_mm              Climática    Muy Alta 1.0000 Seleccionada
       4                                                          Radiacion_Solar              Climática    Muy Alta 1.0000 Seleccionada
       5                                                                     irca       Calidad del Agua    Muy Alta 1.0000 Seleccionada
       6                                     

## **M3 Variables Objetivo**

In [2]:
#==========================================================================================
# M3. VARIABLES OBJETIVO DEL DISEÑO EXPERIMENTAL
#==========================================================================================

import pandas as pd

URL_DATASET_MAESTRO = (
    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/"
    "refs/heads/main/DATA/MASTER/C09_MASTER/"
    "Dataset_Maestro_Framework_v04.parquet"
)

dataset_maestro = pd.read_parquet(URL_DATASET_MAESTRO)

#------------------------------------------------------------------------------------------
# Variables objetivo definidas por los experimentos
#------------------------------------------------------------------------------------------

tabla_objetivo = pd.DataFrame([
    ["Exp01", "irca",                                  "Clasificación"],
    ["Exp02", "CALIDAD DEL AGUA",                      "Clasificación"],
    ["Exp03", "Nivel_Minimo",                          "Regresión"],
    ["Exp04", "VolumenUtilDiarioMasa",                 "Regresión"],
    ["Exp05", "DEMANDA BIOQUIMICA DE OXIGENO (DBO5)", "Regresión"],
    ["Exp06", "DEMANDA QUIMICA DE OXIGENO (DQO)",      "Regresión"],
    ["Exp07", "OXIGENO DISUELTO (OD)",                 "Regresión"],
    ["Exp08", "pH",                                    "Regresión"],
], columns=[
    "Experimento",
    "Variable_Objetivo",
    "Tipo_Problema"
])

tabla_objetivo.insert(
    0,
    "Orden",
    range(1, len(tabla_objetivo) + 1)
)

#------------------------------------------------------------------------------------------
# Validación contra Dataset Maestro V04
#------------------------------------------------------------------------------------------

tabla_objetivo["Existe_Dataset"] = tabla_objetivo[
    "Variable_Objetivo"
].isin(dataset_maestro.columns)

tabla_objetivo["Registros_Disponibles"] = tabla_objetivo[
    "Variable_Objetivo"
].apply(
    lambda variable:
        dataset_maestro[variable].notna().sum()
        if variable in dataset_maestro.columns
        else 0
)

tabla_objetivo["Cobertura_%"] = (
    tabla_objetivo["Registros_Disponibles"]
    / len(dataset_maestro)
    * 100
).round(2)

variables_objetivo = tabla_objetivo[
    "Variable_Objetivo"
].tolist()

print()
print("=" * 90)
print("M3 - VARIABLES OBJETIVO")
print("=" * 90)

print()
print("Variables objetivo :", len(tabla_objetivo))
print("Variables únicas   :", tabla_objetivo["Variable_Objetivo"].nunique())
print("Encontradas V04    :", tabla_objetivo["Existe_Dataset"].sum())
print("Faltantes V04      :", (~tabla_objetivo["Existe_Dataset"]).sum())

print()

print(
    tabla_objetivo[
        [
            "Experimento",
            "Variable_Objetivo",
            "Tipo_Problema",
            "Existe_Dataset",
            "Registros_Disponibles",
            "Cobertura_%"
        ]
    ].to_string(index=False)
)


M3 - VARIABLES OBJETIVO

Variables objetivo : 8
Variables únicas   : 8
Encontradas V04    : 8
Faltantes V04      : 0

Experimento                    Variable_Objetivo Tipo_Problema  Existe_Dataset  Registros_Disponibles  Cobertura_%
      Exp01                                 irca Clasificación            True                    528       100.00
      Exp02                     CALIDAD DEL AGUA Clasificación            True                     96        18.18
      Exp03                         Nivel_Minimo     Regresión            True                     87        16.48
      Exp04                VolumenUtilDiarioMasa     Regresión            True                    336        63.64
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)     Regresión            True                     26         4.92
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)     Regresión            True                     49         9.28
      Exp07                OXIGENO DISUELTO (OD)     Regresión            Tr

## **M4 Variables Predictoras**


In [3]:
#==========================================================================================
# M4. VARIABLES PREDICTORAS POR EXPERIMENTO
#==========================================================================================

#------------------------------------------------------------------------------------------
# Universo base seleccionado por C11
#------------------------------------------------------------------------------------------

variables_predictoras_base = (
    df_variables.loc[
        df_variables["Estado_Final"] == "Seleccionada",
        "Variable"
    ]
    .drop_duplicates()
    .tolist()
)

#------------------------------------------------------------------------------------------
# Construcción de predictores por experimento
# Se elimina la variable objetivo de X cuando pertenece al universo seleccionado.
#------------------------------------------------------------------------------------------

registros_predictoras = []

for _, experimento in tabla_objetivo.iterrows():

    id_experimento = experimento["Experimento"]
    variable_objetivo = experimento["Variable_Objetivo"]

    predictoras_experimento = [
        variable
        for variable in variables_predictoras_base
        if variable != variable_objetivo
    ]

    for orden, variable in enumerate(predictoras_experimento, start=1):

        metadata = df_variables.loc[
            df_variables["Variable"] == variable
        ].iloc[0]

        registros_predictoras.append({
            "Experimento": id_experimento,
            "Variable_Objetivo": variable_objetivo,
            "Orden_Predictor": orden,
            "Variable_Predictora": variable,
            "Ranking_IPML": metadata["Ranking"],
            "IPML": metadata["IPML"],
            "Categoria": metadata["Categoria"]
        })

tabla_predictoras = pd.DataFrame(registros_predictoras)

#------------------------------------------------------------------------------------------
# Controles
#------------------------------------------------------------------------------------------

resumen_predictoras = (
    tabla_predictoras
    .groupby("Experimento")
    .agg(
        Variable_Objetivo=("Variable_Objetivo", "first"),
        Predictores=("Variable_Predictora", "count"),
        Predictores_Unicos=("Variable_Predictora", "nunique")
    )
    .reset_index()
)

fugas = (
    tabla_predictoras["Variable_Predictora"]
    == tabla_predictoras["Variable_Objetivo"]
).sum()

duplicados = tabla_predictoras.duplicated(
    subset=["Experimento", "Variable_Predictora"]
).sum()

print()
print("=" * 90)
print("M4 - VARIABLES PREDICTORAS POR EXPERIMENTO")
print("=" * 90)

print()
print("Variables base IPML :", len(variables_predictoras_base))
print("Fugas objetivo → X  :", fugas)
print("Duplicados          :", duplicados)

print()
print(resumen_predictoras.to_string(index=False))


M4 - VARIABLES PREDICTORAS POR EXPERIMENTO

Variables base IPML : 10
Fugas objetivo → X  : 0
Duplicados          : 0

Experimento                    Variable_Objetivo  Predictores  Predictores_Unicos
      Exp01                                 irca            9                   9
      Exp02                     CALIDAD DEL AGUA           10                  10
      Exp03                         Nivel_Minimo           10                  10
      Exp04                VolumenUtilDiarioMasa            9                   9
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)           10                  10
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)           10                  10
      Exp07                OXIGENO DISUELTO (OD)           10                  10
      Exp08                                   pH           10                  10


## **M5 – Catálogo de Experimentos**

Definir formalmente los experimentos que serán ejecutados para validar el Framework, estableciendo la variable objetivo, el tipo de problema, la configuración general y el propósito científico de cada uno.

Este módulo constituye el plan experimental oficial del Framework.

In [4]:
#==========================================================================================
# M5. CATÁLOGO DE EXPERIMENTOS
#==========================================================================================

preguntas = {
    "Exp01": "¿Es posible anticipar el riesgo sanitario del agua mediante información climática, hidrológica y de infraestructura?",
    "Exp02": "¿Qué factores permiten anticipar el estado general de la calidad del agua?",
    "Exp03": "¿Es posible anticipar la disminución del nivel mínimo del recurso hídrico?",
    "Exp04": "¿Qué variables permiten anticipar el volumen útil disponible del sistema?",
    "Exp05": "¿Es posible anticipar procesos de contaminación orgánica del agua?",
    "Exp06": "¿Qué variables permiten anticipar la contaminación química del recurso hídrico?",
    "Exp07": "¿Qué factores permiten anticipar cambios en el oxígeno disuelto del recurso hídrico?",
    "Exp08": "¿Es posible anticipar cambios en el pH del agua?"
}

objetivos_experimento = {
    "Exp01": "Evaluar la capacidad del Framework para predecir el IRCA y analizar factores asociados al riesgo sanitario del agua.",
    "Exp02": "Evaluar la capacidad del Framework para predecir la calidad general del recurso hídrico.",
    "Exp03": "Evaluar la capacidad predictiva del Framework sobre el nivel mínimo del recurso hídrico.",
    "Exp04": "Evaluar la capacidad del Framework para predecir el volumen útil disponible.",
    "Exp05": "Evaluar la capacidad predictiva del Framework sobre la Demanda Bioquímica de Oxígeno (DBO5).",
    "Exp06": "Evaluar la capacidad predictiva del Framework sobre la Demanda Química de Oxígeno (DQO).",
    "Exp07": "Evaluar la capacidad predictiva del Framework sobre el Oxígeno Disuelto (OD).",
    "Exp08": "Evaluar la capacidad predictiva del Framework sobre el pH."
}

#------------------------------------------------------------------------------------------
# Construcción desde M3 para garantizar consistencia de nombres
#------------------------------------------------------------------------------------------

experimentos = tabla_objetivo[
    [
        "Experimento",
        "Variable_Objetivo",
        "Tipo_Problema",
        "Registros_Disponibles",
        "Cobertura_%"
    ]
].copy()

experimentos["Dominio"] = "Gestión Hídrica"

experimentos["Pregunta_Investigacion"] = (
    experimentos["Experimento"].map(preguntas)
)

experimentos["Objetivo_Experimento"] = (
    experimentos["Experimento"].map(objetivos_experimento)
)

experimentos["Ventana"] = 12
experimentos["Horizonte"] = 1
experimentos["Modelo"] = "LSTM"
experimentos["Estado"] = "Pendiente"

# Número de predictores definido en M4
mapa_predictores = resumen_predictoras.set_index(
    "Experimento"
)["Predictores"]

experimentos["Numero_Predictores"] = (
    experimentos["Experimento"].map(mapa_predictores)
)

#------------------------------------------------------------------------------------------
# Orden de columnas
#------------------------------------------------------------------------------------------

experimentos = experimentos[
    [
        "Experimento",
        "Dominio",
        "Pregunta_Investigacion",
        "Objetivo_Experimento",
        "Variable_Objetivo",
        "Tipo_Problema",
        "Numero_Predictores",
        "Registros_Disponibles",
        "Cobertura_%",
        "Ventana",
        "Horizonte",
        "Modelo",
        "Estado"
    ]
]

#------------------------------------------------------------------------------------------
# Control
#------------------------------------------------------------------------------------------

print()
print("=" * 90)
print("M5 - CATÁLOGO DE EXPERIMENTOS")
print("=" * 90)

print()
print("Experimentos       :", len(experimentos))
print("Experimentos únicos:", experimentos["Experimento"].nunique())
print("Objetivos únicos   :", experimentos["Variable_Objetivo"].nunique())
print("Nulos catálogo     :", experimentos.isna().sum().sum())

print()

print(
    experimentos[
        [
            "Experimento",
            "Variable_Objetivo",
            "Tipo_Problema",
            "Numero_Predictores",
            "Cobertura_%",
            "Estado"
        ]
    ].to_string(index=False)
)


M5 - CATÁLOGO DE EXPERIMENTOS

Experimentos       : 8
Experimentos únicos: 8
Objetivos únicos   : 8
Nulos catálogo     : 0

Experimento                    Variable_Objetivo Tipo_Problema  Numero_Predictores  Cobertura_%    Estado
      Exp01                                 irca Clasificación                   9       100.00 Pendiente
      Exp02                     CALIDAD DEL AGUA Clasificación                  10        18.18 Pendiente
      Exp03                         Nivel_Minimo     Regresión                  10        16.48 Pendiente
      Exp04                VolumenUtilDiarioMasa     Regresión                   9        63.64 Pendiente
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)     Regresión                  10         4.92 Pendiente
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)     Regresión                  10         9.28 Pendiente
      Exp07                OXIGENO DISUELTO (OD)     Regresión                  10         9.47 Pendiente
      Exp08                

## **M6 – Configuración Experimental**

In [5]:
#==========================================================================================
# M6. VIABILIDAD EXPERIMENTAL
#==========================================================================================

UMBRAL_COBERTURA = 50.0

#------------------------------------------------------------------------------------------
# Evaluación de viabilidad según cobertura de la variable objetivo
#------------------------------------------------------------------------------------------

experimentos["Umbral_Cobertura_%"] = UMBRAL_COBERTURA

experimentos["Viabilidad"] = (
    experimentos["Cobertura_%"]
    .ge(UMBRAL_COBERTURA)
    .map({
        True: "Viable",
        False: "No viable"
    })
)

experimentos["Motivo_Viabilidad"] = experimentos.apply(
    lambda fila:
        "Cobertura de la variable objetivo suficiente para ejecutar el experimento."
        if fila["Cobertura_%"] >= UMBRAL_COBERTURA
        else
        "Cobertura de la variable objetivo inferior al umbral mínimo del 50%.",
    axis=1
)

#------------------------------------------------------------------------------------------
# Control
#------------------------------------------------------------------------------------------

print()
print("=" * 90)
print("M6 - VIABILIDAD EXPERIMENTAL")
print("=" * 90)

print()
print("Umbral de cobertura :", f"{UMBRAL_COBERTURA:.2f}%")
print("Experimentos        :", len(experimentos))
print("Viables             :", (experimentos["Viabilidad"] == "Viable").sum())
print("No viables          :", (experimentos["Viabilidad"] == "No viable").sum())

print()

print(
    experimentos[
        [
            "Experimento",
            "Variable_Objetivo",
            "Registros_Disponibles",
            "Cobertura_%",
            "Umbral_Cobertura_%",
            "Viabilidad"
        ]
    ].to_string(index=False)
)


M6 - VIABILIDAD EXPERIMENTAL

Umbral de cobertura : 50.00%
Experimentos        : 8
Viables             : 2
No viables          : 6

Experimento                    Variable_Objetivo  Registros_Disponibles  Cobertura_%  Umbral_Cobertura_% Viabilidad
      Exp01                                 irca                    528       100.00                50.0     Viable
      Exp02                     CALIDAD DEL AGUA                     96        18.18                50.0  No viable
      Exp03                         Nivel_Minimo                     87        16.48                50.0  No viable
      Exp04                VolumenUtilDiarioMasa                    336        63.64                50.0     Viable
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)                     26         4.92                50.0  No viable
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)                     49         9.28                50.0  No viable
      Exp07                OXIGENO DISUELTO (OD)       

## **M7 – Criterios de Evaluación**

In [6]:
#==========================================================================================
# M7. CRITERIOS DE EVALUACIÓN
# CLASIFICACIÓN DE EXPERIMENTOS POR TIPO DE PROBLEMA
#==========================================================================================

clasificacion = experimentos.loc[
    experimentos["Tipo_Problema"] == "Clasificación"
].copy()

regresion = experimentos.loc[
    experimentos["Tipo_Problema"] == "Regresión"
].copy()

print()
print("=" * 90)
print("M7 - CRITERIOS DE EVALUACIÓN")
print("=" * 90)

print()
print("Experimentos de Clasificación :", len(clasificacion))
print("Experimentos de Regresión     :", len(regresion))

print()
print("VIABILIDAD")
print("-" * 90)

print(
    "Clasificación viables         :",
    (clasificacion["Viabilidad"] == "Viable").sum()
)

print(
    "Regresión viables             :",
    (regresion["Viabilidad"] == "Viable").sum()
)

print()
print("Experimentos viables:")

print(
    experimentos.loc[
        experimentos["Viabilidad"] == "Viable",
        [
            "Experimento",
            "Variable_Objetivo",
            "Tipo_Problema",
            "Cobertura_%"
        ]
    ].to_string(index=False)
)


M7 - CRITERIOS DE EVALUACIÓN

Experimentos de Clasificación : 2
Experimentos de Regresión     : 6

VIABILIDAD
------------------------------------------------------------------------------------------
Clasificación viables         : 1
Regresión viables             : 1

Experimentos viables:
Experimento     Variable_Objetivo Tipo_Problema  Cobertura_%
      Exp01                  irca Clasificación       100.00
      Exp04 VolumenUtilDiarioMasa     Regresión        63.64


In [7]:
#==========================================================================================
# M7.1 CRITERIOS DE EVALUACIÓN PARA CLASIFICACIÓN
#==========================================================================================

criterios_clasificacion = pd.DataFrame({

    "Metrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Accuracy Balanceada"
    ],

    "Valor_Minimo": [
        0.85,
        0.70,
        0.70,
        0.70,
        0.70
    ],

    "Descripcion": [
        "Exactitud global",
        "Capacidad para evitar falsos positivos",
        "Capacidad para detectar positivos",
        "Equilibrio entre Precision y Recall",
        "Control del desbalance"
    ]

})

print()
print("=" * 90)
print("M7.1 - CRITERIOS DE EVALUACIÓN PARA CLASIFICACIÓN")
print("=" * 90)
print()

print(criterios_clasificacion.to_string(index=False))

print()
print("Criterios definidos :", len(criterios_clasificacion))
print("Métricas únicas     :", criterios_clasificacion["Metrica"].nunique())
print("Nulos               :", criterios_clasificacion.isna().sum().sum())


M7.1 - CRITERIOS DE EVALUACIÓN PARA CLASIFICACIÓN

            Metrica  Valor_Minimo                            Descripcion
           Accuracy          0.85                       Exactitud global
          Precision          0.70 Capacidad para evitar falsos positivos
             Recall          0.70      Capacidad para detectar positivos
           F1-Score          0.70    Equilibrio entre Precision y Recall
Accuracy Balanceada          0.70                 Control del desbalance

Criterios definidos : 5
Métricas únicas     : 5
Nulos               : 0


In [8]:
#==========================================================================================
# M7.2 CRITERIOS DE EVALUACIÓN PARA REGRESIÓN
#==========================================================================================

criterios_regresion = pd.DataFrame({

    "Metrica": [
        "MAE",
        "RMSE",
        "R2"
    ],

    "Criterio": [
        "Minimizar",
        "Minimizar",
        "Mayor o igual a 0.70"
    ],

    "Descripcion": [
        "Error Absoluto Medio",
        "Raíz del Error Cuadrático Medio",
        "Coeficiente de Determinación"
    ]

})

print()
print("=" * 90)
print("M7.2 - CRITERIOS DE EVALUACIÓN PARA REGRESIÓN")
print("=" * 90)
print()

print(criterios_regresion.to_string(index=False))

print()
print("Criterios definidos :", len(criterios_regresion))
print("Métricas únicas     :", criterios_regresion["Metrica"].nunique())
print("Nulos               :", criterios_regresion.isna().sum().sum())


M7.2 - CRITERIOS DE EVALUACIÓN PARA REGRESIÓN

Metrica             Criterio                     Descripcion
    MAE            Minimizar            Error Absoluto Medio
   RMSE            Minimizar Raíz del Error Cuadrático Medio
     R2 Mayor o igual a 0.70    Coeficiente de Determinación

Criterios definidos : 3
Métricas únicas     : 3
Nulos               : 0


In [9]:
#==========================================================================================
# M7.3 REGLAS DE EVALUACIÓN DEL FRAMEWORK
#==========================================================================================

reglas_framework = [

    "Accuracy >= 0.85",
    "Precision >= 0.70",
    "Recall >= 0.70",
    "F1 >= 0.70",
    "Accuracy Balanceada >= 0.70"

]

print()
print("=" * 90)
print("M7.3 - REGLAS DE EVALUACIÓN DEL FRAMEWORK")
print("=" * 90)

print()

for regla in reglas_framework:
    print("•", regla)

print()
print("Reglas definidas :", len(reglas_framework))


M7.3 - REGLAS DE EVALUACIÓN DEL FRAMEWORK

• Accuracy >= 0.85
• Precision >= 0.70
• Recall >= 0.70
• F1 >= 0.70
• Accuracy Balanceada >= 0.70

Reglas definidas : 5


## **M8 – Estado de los Experimentos**

In [10]:
#==========================================================================================
# M8. ESTADO DE LOS EXPERIMENTOS
#==========================================================================================

import numpy as np

estado_experimentos = experimentos.copy()

#------------------------------------------------------------------------------------------
# Campos reservados para resultados posteriores
#------------------------------------------------------------------------------------------

estado_experimentos["Estado_Modelo"] = ""

estado_experimentos["Accuracy"] = np.nan
estado_experimentos["Precision"] = np.nan
estado_experimentos["Recall"] = np.nan
estado_experimentos["F1"] = np.nan

estado_experimentos["Fecha_Ejecucion"] = ""
estado_experimentos["Observaciones"] = ""

#------------------------------------------------------------------------------------------
# Controles
#------------------------------------------------------------------------------------------

print()
print("=" * 90)
print("M8 - ESTADO DE LOS EXPERIMENTOS")
print("=" * 90)

print()
print("Experimentos       :", len(estado_experimentos))
print("Experimentos únicos:", estado_experimentos["Experimento"].nunique())

print(
    "Viables            :",
    (estado_experimentos["Viabilidad"] == "Viable").sum()
)

print(
    "No viables         :",
    (estado_experimentos["Viabilidad"] == "No viable").sum()
)

print()

print(
    estado_experimentos[
        [
            "Experimento",
            "Variable_Objetivo",
            "Tipo_Problema",
            "Cobertura_%",
            "Viabilidad",
            "Estado",
            "Estado_Modelo"
        ]
    ].to_string(index=False)
)


M8 - ESTADO DE LOS EXPERIMENTOS

Experimentos       : 8
Experimentos únicos: 8
Viables            : 2
No viables         : 6

Experimento                    Variable_Objetivo Tipo_Problema  Cobertura_% Viabilidad    Estado Estado_Modelo
      Exp01                                 irca Clasificación       100.00     Viable Pendiente              
      Exp02                     CALIDAD DEL AGUA Clasificación        18.18  No viable Pendiente              
      Exp03                         Nivel_Minimo     Regresión        16.48  No viable Pendiente              
      Exp04                VolumenUtilDiarioMasa     Regresión        63.64     Viable Pendiente              
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)     Regresión         4.92  No viable Pendiente              
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)     Regresión         9.28  No viable Pendiente              
      Exp07                OXIGENO DISUELTO (OD)     Regresión         9.47  No viable Pendiente

In [11]:
#==========================================================================================
# M8.1 EXPORTACIÓN DEL ESTADO DE LOS EXPERIMENTOS
#==========================================================================================

estado_experimentos.to_csv(
    "estado_experimentos.csv",
    index=False,
    encoding="utf-8-sig"
)

estado_experimentos.to_excel(
    "estado_experimentos.xlsx",
    index=False
)

#------------------------------------------------------------------------------------------
# Validación
#------------------------------------------------------------------------------------------

import os

print()
print("=" * 90)
print("M8.1 - EXPORTACIÓN DEL ESTADO DE LOS EXPERIMENTOS")
print("=" * 90)

print()
print(
    "estado_experimentos.csv  :",
    os.path.exists("estado_experimentos.csv")
)

print(
    "estado_experimentos.xlsx :",
    os.path.exists("estado_experimentos.xlsx")
)

print()
print("Filas    :", len(estado_experimentos))
print("Columnas :", estado_experimentos.shape[1])


M8.1 - EXPORTACIÓN DEL ESTADO DE LOS EXPERIMENTOS

estado_experimentos.csv  : True
estado_experimentos.xlsx : True

Filas    : 8
Columnas : 23


## **M9 – Exportación del Diseño Experimental**

In [12]:
#==========================================================================================
# M9.1 EXPORTACIÓN Y VALIDACIÓN DEL CATÁLOGO EXPERIMENTAL
#==========================================================================================

import os
import pandas as pd

CARPETA_SALIDA = "DISENO_EXPERIMENTAL"

os.makedirs(
    CARPETA_SALIDA,
    exist_ok=True
)

ruta_catalogo_csv = os.path.join(
    CARPETA_SALIDA,
    "catalogo_experimentos.csv"
)

ruta_catalogo_xlsx = os.path.join(
    CARPETA_SALIDA,
    "catalogo_experimentos.xlsx"
)

#------------------------------------------------------------------------------------------
# Exportación desde el catálogo vigente en memoria
#------------------------------------------------------------------------------------------

experimentos.to_csv(
    ruta_catalogo_csv,
    index=False,
    encoding="utf-8-sig"
)

experimentos.to_excel(
    ruta_catalogo_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Recarga para validar el archivo que realmente consumirá C13
#------------------------------------------------------------------------------------------

catalogo_validacion = pd.read_excel(
    ruta_catalogo_xlsx
)

columnas_c13 = [
    "Experimento",
    "Variable_Objetivo",
    "Tipo_Problema",
    "Ventana",
    "Horizonte",
    "Modelo",
    "Estado"
]

faltantes_c13 = [
    columna
    for columna in columnas_c13
    if columna not in catalogo_validacion.columns
]

print()
print("=" * 90)
print("M9.1 - CATÁLOGO EXPERIMENTAL")
print("=" * 90)

print()
print("CSV generado              :", os.path.exists(ruta_catalogo_csv))
print("XLSX generado             :", os.path.exists(ruta_catalogo_xlsx))
print("Experimentos              :", len(catalogo_validacion))
print("Experimentos únicos       :", catalogo_validacion["Experimento"].nunique())
print("Columnas                  :", catalogo_validacion.shape[1])
print("Columnas requeridas C13   :", len(columnas_c13))
print("Columnas faltantes C13    :", len(faltantes_c13))

if faltantes_c13:
    print("Faltantes:", faltantes_c13)

print()
print(
    catalogo_validacion[
        [
            "Experimento",
            "Variable_Objetivo",
            "Tipo_Problema",
            "Viabilidad",
            "Estado"
        ]
    ].to_string(index=False)
)


M9.1 - CATÁLOGO EXPERIMENTAL

CSV generado              : True
XLSX generado             : True
Experimentos              : 8
Experimentos únicos       : 8
Columnas                  : 16
Columnas requeridas C13   : 7
Columnas faltantes C13    : 0

Experimento                    Variable_Objetivo Tipo_Problema Viabilidad    Estado
      Exp01                                 irca Clasificación     Viable Pendiente
      Exp02                     CALIDAD DEL AGUA Clasificación  No viable Pendiente
      Exp03                         Nivel_Minimo     Regresión  No viable Pendiente
      Exp04                VolumenUtilDiarioMasa     Regresión     Viable Pendiente
      Exp05 DEMANDA BIOQUIMICA DE OXIGENO (DBO5)     Regresión  No viable Pendiente
      Exp06     DEMANDA QUIMICA DE OXIGENO (DQO)     Regresión  No viable Pendiente
      Exp07                OXIGENO DISUELTO (OD)     Regresión  No viable Pendiente
      Exp08                                   pH     Regresión  No viable Pendi

In [ ]:
#==========================================================
# M9 - EXPORTACIÓN DEL DISEÑO EXPERIMENTAL
#==========================================================

import os

CARPETA_SALIDA = "DISEÑO_EXPERIMENTAL"

os.makedirs(

    CARPETA_SALIDA,

    exist_ok=True

)

print()

print("="*90)
print("EXPORTACIÓN DEL DISEÑO EXPERIMENTAL")
print("="*90)

print()

print(f"Carpeta creada: {CARPETA_SALIDA}")


EXPORTACIÓN DEL DISEÑO EXPERIMENTAL

Carpeta creada: DISEÑO_EXPERIMENTAL


In [ ]:
import shutil

archivos = [

"catalogo_experimentos.csv",
"catalogo_experimentos.xlsx",

"configuracion_experimentos.csv",
"configuracion_experimentos.xlsx",

"criterios_clasificacion.csv",
"criterios_clasificacion.xlsx",

"criterios_regresion.csv",
"criterios_regresion.xlsx",

"estado_experimentos.csv",
"estado_experimentos.xlsx",

"variables_predictoras.csv",
"variables_predictoras.xlsx",

"variables_objetivo.csv",
"variables_objetivo.xlsx"

]

for archivo in archivos:

    if os.path.exists(archivo):

        shutil.copy(

            archivo,

            os.path.join(

                CARPETA_SALIDA,

                archivo

            )

        )

print()

print("Archivos exportados correctamente.")


Archivos exportados correctamente.


In [ ]:
readme = f"""
# DISEÑO EXPERIMENTAL DEL FRAMEWORK V7

Este directorio contiene la configuración oficial utilizada por el Framework.

Contenido

• Variables objetivo

• Variables predictoras

• Catálogo experimental

• Configuración experimental

• Criterios de evaluación

• Estado de los experimentos

Estos archivos son consumidos automáticamente por:

FW7_C13_MachineLearning.ipynb

FW7_C14_Modelado.ipynb

FW7_C15_Evaluacion_Modelos.ipynb

"""

with open(

    os.path.join(

        CARPETA_SALIDA,

        "README.md"

    ),

    "w",

    encoding="utf-8"

) as f:

    f.write(readme)

print("README generado.")

README generado.


In [13]:
#==========================================================================================
# M9.2 EXPORTACIÓN DE VARIABLES OBJETIVO Y PREDICTORAS
#==========================================================================================

#------------------------------------------------------------------------------------------
# Variables objetivo
#------------------------------------------------------------------------------------------

ruta_objetivo_csv = os.path.join(
    CARPETA_SALIDA,
    "variables_objetivo.csv"
)

ruta_objetivo_xlsx = os.path.join(
    CARPETA_SALIDA,
    "variables_objetivo.xlsx"
)

tabla_objetivo.to_csv(
    ruta_objetivo_csv,
    index=False,
    encoding="utf-8-sig"
)

tabla_objetivo.to_excel(
    ruta_objetivo_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Variables predictoras por experimento
#------------------------------------------------------------------------------------------

ruta_predictoras_csv = os.path.join(
    CARPETA_SALIDA,
    "variables_predictoras.csv"
)

ruta_predictoras_xlsx = os.path.join(
    CARPETA_SALIDA,
    "variables_predictoras.xlsx"
)

tabla_predictoras.to_csv(
    ruta_predictoras_csv,
    index=False,
    encoding="utf-8-sig"
)

tabla_predictoras.to_excel(
    ruta_predictoras_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Recarga y validación
#------------------------------------------------------------------------------------------

objetivos_validacion = pd.read_excel(
    ruta_objetivo_xlsx
)

predictoras_validacion = pd.read_excel(
    ruta_predictoras_xlsx
)

fugas_validacion = (
    predictoras_validacion["Variable_Predictora"]
    ==
    predictoras_validacion["Variable_Objetivo"]
).sum()

duplicados_predictoras = predictoras_validacion.duplicated(
    subset=["Experimento", "Variable_Predictora"]
).sum()

print()
print("=" * 90)
print("M9.2 - VARIABLES OBJETIVO Y PREDICTORAS")
print("=" * 90)

print()
print("OBJETIVOS")
print("-" * 90)

print("CSV generado        :", os.path.exists(ruta_objetivo_csv))
print("XLSX generado       :", os.path.exists(ruta_objetivo_xlsx))
print("Variables objetivo  :", len(objetivos_validacion))
print("Objetivos únicos    :", objetivos_validacion["Variable_Objetivo"].nunique())

print()
print("PREDICTORAS")
print("-" * 90)

print("CSV generado        :", os.path.exists(ruta_predictoras_csv))
print("XLSX generado       :", os.path.exists(ruta_predictoras_xlsx))
print("Registros           :", len(predictoras_validacion))
print("Fugas objetivo → X  :", fugas_validacion)
print("Duplicados          :", duplicados_predictoras)

print()

print(
    predictoras_validacion
    .groupby("Experimento")
    ["Variable_Predictora"]
    .count()
    .rename("Predictores")
    .to_string()
)


M9.2 - VARIABLES OBJETIVO Y PREDICTORAS

OBJETIVOS
------------------------------------------------------------------------------------------
CSV generado        : True
XLSX generado       : True
Variables objetivo  : 8
Objetivos únicos    : 8

PREDICTORAS
------------------------------------------------------------------------------------------
CSV generado        : True
XLSX generado       : True
Registros           : 78
Fugas objetivo → X  : 0
Duplicados          : 0

Experimento
Exp01     9
Exp02    10
Exp03    10
Exp04     9
Exp05    10
Exp06    10
Exp07    10
Exp08    10


In [14]:
#==========================================================================================
# M9.3 EXPORTACIÓN DE CRITERIOS DE EVALUACIÓN
#==========================================================================================

ruta_criterios_clas_csv = os.path.join(
    CARPETA_SALIDA,
    "criterios_clasificacion.csv"
)

ruta_criterios_clas_xlsx = os.path.join(
    CARPETA_SALIDA,
    "criterios_clasificacion.xlsx"
)

ruta_criterios_reg_csv = os.path.join(
    CARPETA_SALIDA,
    "criterios_regresion.csv"
)

ruta_criterios_reg_xlsx = os.path.join(
    CARPETA_SALIDA,
    "criterios_regresion.xlsx"
)

#------------------------------------------------------------------------------------------
# Clasificación
#------------------------------------------------------------------------------------------

criterios_clasificacion.to_csv(
    ruta_criterios_clas_csv,
    index=False,
    encoding="utf-8-sig"
)

criterios_clasificacion.to_excel(
    ruta_criterios_clas_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Regresión
#------------------------------------------------------------------------------------------

criterios_regresion.to_csv(
    ruta_criterios_reg_csv,
    index=False,
    encoding="utf-8-sig"
)

criterios_regresion.to_excel(
    ruta_criterios_reg_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Validación
#------------------------------------------------------------------------------------------

clas_validacion = pd.read_excel(
    ruta_criterios_clas_xlsx
)

reg_validacion = pd.read_excel(
    ruta_criterios_reg_xlsx
)

print()
print("=" * 90)
print("M9.3 - CRITERIOS DE EVALUACIÓN")
print("=" * 90)

print()
print("CLASIFICACIÓN")
print("-" * 90)

print("CSV generado    :", os.path.exists(ruta_criterios_clas_csv))
print("XLSX generado   :", os.path.exists(ruta_criterios_clas_xlsx))
print("Criterios       :", len(clas_validacion))

print()
print("REGRESIÓN")
print("-" * 90)

print("CSV generado    :", os.path.exists(ruta_criterios_reg_csv))
print("XLSX generado   :", os.path.exists(ruta_criterios_reg_xlsx))
print("Criterios       :", len(reg_validacion))


M9.3 - CRITERIOS DE EVALUACIÓN

CLASIFICACIÓN
------------------------------------------------------------------------------------------
CSV generado    : True
XLSX generado   : True
Criterios       : 5

REGRESIÓN
------------------------------------------------------------------------------------------
CSV generado    : True
XLSX generado   : True
Criterios       : 3


In [15]:
#==========================================================================================
# M9.4 EXPORTACIÓN DEL ESTADO DE LOS EXPERIMENTOS
#==========================================================================================

ruta_estado_csv = os.path.join(
    CARPETA_SALIDA,
    "estado_experimentos.csv"
)

ruta_estado_xlsx = os.path.join(
    CARPETA_SALIDA,
    "estado_experimentos.xlsx"
)

#------------------------------------------------------------------------------------------
# Exportación directa desde el estado vigente en memoria
#------------------------------------------------------------------------------------------

estado_experimentos.to_csv(
    ruta_estado_csv,
    index=False,
    encoding="utf-8-sig"
)

estado_experimentos.to_excel(
    ruta_estado_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Recarga y validación
#------------------------------------------------------------------------------------------

estado_validacion = pd.read_excel(
    ruta_estado_xlsx
)

print()
print("=" * 90)
print("M9.4 - ESTADO DE LOS EXPERIMENTOS")
print("=" * 90)

print()
print("CSV generado        :", os.path.exists(ruta_estado_csv))
print("XLSX generado       :", os.path.exists(ruta_estado_xlsx))
print("Experimentos        :", len(estado_validacion))
print("Experimentos únicos :", estado_validacion["Experimento"].nunique())

print(
    "Viables             :",
    (estado_validacion["Viabilidad"] == "Viable").sum()
)

print(
    "No viables          :",
    (estado_validacion["Viabilidad"] == "No viable").sum()
)

print(
    "Pendientes          :",
    (estado_validacion["Estado"] == "Pendiente").sum()
)

print()
print("Columnas             :", estado_validacion.shape[1])


M9.4 - ESTADO DE LOS EXPERIMENTOS

CSV generado        : True
XLSX generado       : True
Experimentos        : 8
Experimentos únicos : 8
Viables             : 2
No viables          : 6
Pendientes          : 8

Columnas             : 23


In [16]:
#==========================================================================================
# M9.5 CONFIGURACIÓN EXPERIMENTAL
# Recreación compatible con la estructura original
#==========================================================================================

configuracion = experimentos[
    [
        "Experimento",
        "Tipo_Problema"
    ]
].copy()

#------------------------------------------------------------------------------------------
# Parámetros originales del Framework
#------------------------------------------------------------------------------------------

configuracion["Modelo"] = "LSTM"
configuracion["Ventana"] = 12
configuracion["Horizonte"] = 1
configuracion["Transformacion"] = "MinMaxScaler"
configuracion["Optimizador"] = "Adam"
configuracion["LearningRate"] = 0.001
configuracion["BatchSize"] = 32
configuracion["Epochs"] = 100
configuracion["EarlyStopping"] = 10

#------------------------------------------------------------------------------------------
# Loss y métrica según el tipo de problema original
#------------------------------------------------------------------------------------------

configuracion["Loss"] = configuracion["Tipo_Problema"].map({
    "Clasificación": "binary_crossentropy",
    "Regresión": "mse"
})

configuracion["Metrica"] = configuracion["Tipo_Problema"].map({
    "Clasificación": "Accuracy",
    "Regresión": "RMSE"
})

# Tipo_Problema se usa únicamente para construir la configuración;
# no formaba parte del archivo original.
configuracion = configuracion.drop(
    columns=["Tipo_Problema"]
)

#------------------------------------------------------------------------------------------
# Orden original de columnas
#------------------------------------------------------------------------------------------

configuracion = configuracion[
    [
        "Experimento",
        "Modelo",
        "Ventana",
        "Horizonte",
        "Transformacion",
        "Optimizador",
        "LearningRate",
        "BatchSize",
        "Epochs",
        "EarlyStopping",
        "Loss",
        "Metrica"
    ]
]

#------------------------------------------------------------------------------------------
# Exportación
#------------------------------------------------------------------------------------------

ruta_config_csv = os.path.join(
    CARPETA_SALIDA,
    "configuracion_experimentos.csv"
)

ruta_config_xlsx = os.path.join(
    CARPETA_SALIDA,
    "configuracion_experimentos.xlsx"
)

configuracion.to_csv(
    ruta_config_csv,
    index=False,
    encoding="utf-8-sig"
)

configuracion.to_excel(
    ruta_config_xlsx,
    index=False
)

#------------------------------------------------------------------------------------------
# Validación
#------------------------------------------------------------------------------------------

config_validacion = pd.read_excel(
    ruta_config_xlsx
)

print()
print("=" * 90)
print("M9.5 - CONFIGURACIÓN EXPERIMENTAL")
print("=" * 90)

print()
print("CSV generado        :", os.path.exists(ruta_config_csv))
print("XLSX generado       :", os.path.exists(ruta_config_xlsx))
print("Experimentos        :", len(config_validacion))
print("Experimentos únicos :", config_validacion["Experimento"].nunique())
print("Columnas            :", config_validacion.shape[1])
print("Nulos               :", config_validacion.isna().sum().sum())

print()
print(
    config_validacion[
        [
            "Experimento",
            "Modelo",
            "Ventana",
            "Horizonte",
            "Loss",
            "Metrica"
        ]
    ].to_string(index=False)
)


M9.5 - CONFIGURACIÓN EXPERIMENTAL

CSV generado        : True
XLSX generado       : True
Experimentos        : 8
Experimentos únicos : 8
Columnas            : 12
Nulos               : 0

Experimento Modelo  Ventana  Horizonte                Loss  Metrica
      Exp01   LSTM       12          1 binary_crossentropy Accuracy
      Exp02   LSTM       12          1 binary_crossentropy Accuracy
      Exp03   LSTM       12          1                 mse     RMSE
      Exp04   LSTM       12          1                 mse     RMSE
      Exp05   LSTM       12          1                 mse     RMSE
      Exp06   LSTM       12          1                 mse     RMSE
      Exp07   LSTM       12          1                 mse     RMSE
      Exp08   LSTM       12          1                 mse     RMSE


In [17]:
#==========================================================================================
# M9.6 VALIDACIÓN INTEGRAL DEL PAQUETE DE DISEÑO EXPERIMENTAL
#==========================================================================================

archivos_esperados = [

    "catalogo_experimentos.csv",
    "catalogo_experimentos.xlsx",

    "configuracion_experimentos.csv",
    "configuracion_experimentos.xlsx",

    "criterios_clasificacion.csv",
    "criterios_clasificacion.xlsx",

    "criterios_regresion.csv",
    "criterios_regresion.xlsx",

    "estado_experimentos.csv",
    "estado_experimentos.xlsx",

    "variables_predictoras.csv",
    "variables_predictoras.xlsx",

    "variables_objetivo.csv",
    "variables_objetivo.xlsx"
]

archivos_encontrados = sorted(
    [
        archivo
        for archivo in os.listdir(CARPETA_SALIDA)
        if os.path.isfile(
            os.path.join(CARPETA_SALIDA, archivo)
        )
    ]
)

faltantes = [
    archivo
    for archivo in archivos_esperados
    if archivo not in archivos_encontrados
]

print()
print("=" * 90)
print("M9.6 - VALIDACIÓN INTEGRAL DEL DISEÑO EXPERIMENTAL")
print("=" * 90)

print()
print("Carpeta                    :", CARPETA_SALIDA)
print("Archivos esperados         :", len(archivos_esperados))
print("Archivos esperados presentes:", len(archivos_esperados) - len(faltantes))
print("Archivos faltantes         :", len(faltantes))

print()

for archivo in archivos_esperados:

    ruta = os.path.join(
        CARPETA_SALIDA,
        archivo
    )

    estado = "OK" if os.path.exists(ruta) else "FALTA"

    print(f"{estado:5} | {archivo}")

if faltantes:

    print()
    print("ATENCIÓN - Archivos faltantes:")

    for archivo in faltantes:
        print(" -", archivo)

else:

    print()
    print("PAQUETE DE DISEÑO EXPERIMENTAL COMPLETO")


M9.6 - VALIDACIÓN INTEGRAL DEL DISEÑO EXPERIMENTAL

Carpeta                    : DISENO_EXPERIMENTAL
Archivos esperados         : 14
Archivos esperados presentes: 14
Archivos faltantes         : 0

OK    | catalogo_experimentos.csv
OK    | catalogo_experimentos.xlsx
OK    | configuracion_experimentos.csv
OK    | configuracion_experimentos.xlsx
OK    | criterios_clasificacion.csv
OK    | criterios_clasificacion.xlsx
OK    | criterios_regresion.csv
OK    | criterios_regresion.xlsx
OK    | estado_experimentos.csv
OK    | estado_experimentos.xlsx
OK    | variables_predictoras.csv
OK    | variables_predictoras.xlsx
OK    | variables_objetivo.csv
OK    | variables_objetivo.xlsx

PAQUETE DE DISEÑO EXPERIMENTAL COMPLETO


In [18]:
#==========================================================================================
# M9.7 README DEL DISEÑO EXPERIMENTAL
#==========================================================================================

ruta_readme = os.path.join(
    CARPETA_SALIDA,
    "README.md"
)

readme = """
# DISEÑO EXPERIMENTAL DEL FRAMEWORK V7

Este directorio contiene los artefactos oficiales del Diseño Experimental
utilizados por el Framework V7.

## Contenido

- Variables objetivo
- Variables predictoras
- Catálogo experimental
- Configuración experimental
- Criterios de evaluación
- Estado de los experimentos

## Experimentos

El diseño contiene 8 experimentos:

- Exp01 a Exp08

Los experimentos conservan su definición independientemente de su
viabilidad actual.

La viabilidad se determina a partir de la cobertura disponible de la
variable objetivo, utilizando un umbral de referencia del 50%.

## Integración

Estos artefactos forman parte del flujo experimental del Framework V7
y sirven como insumo para las etapas posteriores de Machine Learning
y para su integración con FRAMEWORK_STREAMLIT.

"""

with open(
    ruta_readme,
    "w",
    encoding="utf-8"
) as f:
    f.write(readme.strip() + "\n")

#------------------------------------------------------------------------------------------
# Validación
#------------------------------------------------------------------------------------------

print()
print("=" * 90)
print("M9.7 - README DEL DISEÑO EXPERIMENTAL")
print("=" * 90)

print()
print("README generado :", os.path.exists(ruta_readme))

with open(
    ruta_readme,
    "r",
    encoding="utf-8"
) as f:
    contenido_readme = f.read()

print("Caracteres       :", len(contenido_readme))
print("Menciona 8 exp.  :", "8 experimentos" in contenido_readme)
print("Menciona Streamlit:", "FRAMEWORK_STREAMLIT" in contenido_readme)


M9.7 - README DEL DISEÑO EXPERIMENTAL

README generado : True
Caracteres       : 806
Menciona 8 exp.  : True
Menciona Streamlit: True


In [19]:
#==========================================================================================
# M9.8 VALIDACIÓN FINAL DE CONSISTENCIA DEL DISEÑO EXPERIMENTAL
#==========================================================================================

import os
import pandas as pd

#------------------------------------------------------------------------------------------
# Recarga de artefactos oficiales exportados
#------------------------------------------------------------------------------------------

cat_final = pd.read_excel(
    os.path.join(CARPETA_SALIDA, "catalogo_experimentos.xlsx")
)

obj_final = pd.read_excel(
    os.path.join(CARPETA_SALIDA, "variables_objetivo.xlsx")
)

pred_final = pd.read_excel(
    os.path.join(CARPETA_SALIDA, "variables_predictoras.xlsx")
)

config_final = pd.read_excel(
    os.path.join(CARPETA_SALIDA, "configuracion_experimentos.xlsx")
)

estado_final = pd.read_excel(
    os.path.join(CARPETA_SALIDA, "estado_experimentos.xlsx")
)

#------------------------------------------------------------------------------------------
# Conjuntos de experimentos
#------------------------------------------------------------------------------------------

exp_catalogo = set(cat_final["Experimento"])
exp_objetivos = set(obj_final["Experimento"])
exp_predictoras = set(pred_final["Experimento"])
exp_config = set(config_final["Experimento"])
exp_estado = set(estado_final["Experimento"])

#------------------------------------------------------------------------------------------
# Validación objetivos Catálogo <-> Variables objetivo
#------------------------------------------------------------------------------------------

comparacion_objetivos = (
    cat_final[
        ["Experimento", "Variable_Objetivo"]
    ]
    .merge(
        obj_final[
            ["Experimento", "Variable_Objetivo"]
        ],
        on="Experimento",
        suffixes=("_Catalogo", "_Objetivos")
    )
)

objetivos_coinciden = (
    comparacion_objetivos["Variable_Objetivo_Catalogo"]
    ==
    comparacion_objetivos["Variable_Objetivo_Objetivos"]
).all()

#------------------------------------------------------------------------------------------
# Validación número de predictores
#------------------------------------------------------------------------------------------

conteo_pred = (
    pred_final
    .groupby("Experimento")
    ["Variable_Predictora"]
    .count()
    .rename("Predictores_Exportados")
    .reset_index()
)

comparacion_pred = cat_final[
    ["Experimento", "Numero_Predictores"]
].merge(
    conteo_pred,
    on="Experimento",
    how="left"
)

numero_predictores_coincide = (
    comparacion_pred["Numero_Predictores"]
    ==
    comparacion_pred["Predictores_Exportados"]
).all()

#------------------------------------------------------------------------------------------
# Validación de fuga objetivo -> X
#------------------------------------------------------------------------------------------

fugas_objetivo = (
    pred_final["Variable_Predictora"]
    ==
    pred_final["Variable_Objetivo"]
).sum()

#------------------------------------------------------------------------------------------
# Compatibilidad Catálogo <-> Configuración
#------------------------------------------------------------------------------------------

comparacion_config = cat_final[
    [
        "Experimento",
        "Modelo",
        "Ventana",
        "Horizonte"
    ]
].merge(
    config_final[
        [
            "Experimento",
            "Modelo",
            "Ventana",
            "Horizonte"
        ]
    ],
    on="Experimento",
    suffixes=("_Catalogo", "_Config")
)

config_coincide = (
    (
        comparacion_config["Modelo_Catalogo"]
        ==
        comparacion_config["Modelo_Config"]
    )
    &
    (
        comparacion_config["Ventana_Catalogo"]
        ==
        comparacion_config["Ventana_Config"]
    )
    &
    (
        comparacion_config["Horizonte_Catalogo"]
        ==
        comparacion_config["Horizonte_Config"]
    )
).all()

#------------------------------------------------------------------------------------------
# Columnas requeridas por C13
#------------------------------------------------------------------------------------------

columnas_c13 = [
    "Experimento",
    "Variable_Objetivo",
    "Tipo_Problema",
    "Ventana",
    "Horizonte",
    "Modelo",
    "Estado"
]

faltantes_c13 = [
    c for c in columnas_c13
    if c not in cat_final.columns
]

#------------------------------------------------------------------------------------------
# Controles finales
#------------------------------------------------------------------------------------------

controles = {

    "Catálogo = 8 experimentos":
        len(cat_final) == 8
        and cat_final["Experimento"].nunique() == 8,

    "Objetivos = 8":
        len(obj_final) == 8
        and obj_final["Variable_Objetivo"].nunique() == 8,

    "Configuraciones = 8":
        len(config_final) == 8
        and config_final["Experimento"].nunique() == 8,

    "Estado = 8":
        len(estado_final) == 8
        and estado_final["Experimento"].nunique() == 8,

    "Predictoras = 78 registros":
        len(pred_final) == 78,

    "Mismos experimentos en artefactos":
        exp_catalogo
        == exp_objetivos
        == exp_predictoras
        == exp_config
        == exp_estado,

    "Objetivos consistentes":
        objetivos_coinciden,

    "Número de predictores consistente":
        numero_predictores_coincide,

    "Sin fuga objetivo -> X":
        fugas_objetivo == 0,

    "Configuración consistente":
        config_coincide,

    "Columnas C13 completas":
        len(faltantes_c13) == 0,

    "Viabilidad 2 / 6":
        (cat_final["Viabilidad"] == "Viable").sum() == 2
        and
        (cat_final["Viabilidad"] == "No viable").sum() == 6,

    "8 experimentos pendientes":
        (cat_final["Estado"] == "Pendiente").sum() == 8,

    "README presente":
        os.path.exists(
            os.path.join(CARPETA_SALIDA, "README.md")
        )
}

#------------------------------------------------------------------------------------------
# Resultado
#------------------------------------------------------------------------------------------

print()
print("=" * 90)
print("M9.8 - VALIDACIÓN FINAL DEL DISEÑO EXPERIMENTAL")
print("=" * 90)
print()

for nombre, resultado in controles.items():

    estado = "OK" if resultado else "ERROR"

    print(f"{estado:5} | {nombre}")

print()
print("Controles ejecutados :", len(controles))
print("Controles OK         :", sum(controles.values()))
print("Controles con error  :", len(controles) - sum(controles.values()))

if all(controles.values()):

    print()
    print("DISEÑO EXPERIMENTAL CONSISTENTE")

else:

    print()
    print("ATENCIÓN - REVISAR ANTES DE CERRAR EL DISEÑO EXPERIMENTAL")


M9.8 - VALIDACIÓN FINAL DEL DISEÑO EXPERIMENTAL

OK    | Catálogo = 8 experimentos
OK    | Objetivos = 8
OK    | Configuraciones = 8
OK    | Estado = 8
OK    | Predictoras = 78 registros
OK    | Mismos experimentos en artefactos
OK    | Objetivos consistentes
OK    | Número de predictores consistente
OK    | Sin fuga objetivo -> X
OK    | Configuración consistente
OK    | Columnas C13 completas
OK    | Viabilidad 2 / 6
OK    | 8 experimentos pendientes
OK    | README presente

Controles ejecutados : 14
Controles OK         : 14
Controles con error  : 0

DISEÑO EXPERIMENTAL CONSISTENTE
